In [0]:
volume_path = "/Volumes/coretec_dev/landing/coretec_files/"

def listar_volume(path, nivel=0):
    itens = dbutils.fs.ls(path)

    for item in itens:
        tipo = "DIRETÓRIO" if item.path.endswith("/") else "ARQUIVO"

        print(
            f"{'  ' * nivel}{tipo} | "
            f"{item.path} | "
            f"{item.size} bytes"
        )

        if item.path.endswith("/"):
            listar_volume(item.path, nivel + 1)

listar_volume(volume_path)

In [0]:
acessos_path = "/Volumes/coretec_dev/landing/coretec_files/acessos_portaria/incoming/"

df_acessos_raw = spark.read.parquet(acessos_path)

print("Quantidade de registros:", df_acessos_raw.count())

df_acessos_raw.printSchema()

display(df_acessos_raw.limit(10))


In [0]:
from pyspark.sql import functions as F

df_acessos_com_origem = df_acessos_raw.withColumn(
    "_source_file",
    F.col("_metadata.file_path")
)

display(
    df_acessos_com_origem
        .groupBy("_source_file")
        .count()
        .orderBy("_source_file")
)

In [0]:
batch_id_acessos = "acessos_portaria_2026_01_05_batch_001"

df_acessos_bronze = (
    df_acessos_com_origem
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_ingestion_date",
            F.current_date()
        )
        .withColumn(
            "_batch_id",
            F.lit(batch_id_acessos)
        )
)

df_acessos_bronze.printSchema()

display(df_acessos_bronze.limit(10))

In [0]:
tabela_acessos = "coretec_dev.bronze.acessos_portaria"

(
    df_acessos_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_acessos)
)

print(f"Tabela criada: {tabela_acessos}")

In [0]:
tabela_acessos = "coretec_dev.bronze.acessos_portaria"

print(
    "Tabela existe:",
    spark.catalog.tableExists(tabela_acessos)
)

print(
    "Quantidade de registros:",
    spark.table(tabela_acessos).count()
)

In [0]:
detalhes_acessos = spark.sql("""
    DESCRIBE DETAIL coretec_dev.bronze.acessos_portaria
""")

display(
    detalhes_acessos.select(
        "format",
        "name",
        "location",
        "numFiles",
        "sizeInBytes"
    )
)

In [0]:
historico_acessos = spark.sql("""
    DESCRIBE HISTORY coretec_dev.bronze.acessos_portaria
""")

display(
    historico_acessos.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)

In [0]:
df_acessos_tabela = spark.table(
    "coretec_dev.bronze.acessos_portaria"
)

df_acessos_tabela.printSchema()

len(df_acessos_tabela.columns)



In [0]:
ocorrencias_path = (
    "/Volumes/coretec_dev/landing/coretec_files/"
    "ocorrencias/incoming/"
)

df_ocorrencias_raw = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(ocorrencias_path)
)

print("DataFrame de ocorrências criado com sucesso.")

In [0]:
from pyspark.sql import functions as F

print("Quantidade total de registros:", df_ocorrencias_raw.count())

print("\nSchema do DataFrame:")
df_ocorrencias_raw.printSchema()

print("\nAmostra dos dados:")
display(df_ocorrencias_raw.limit(10))

print("\nQuantidade de registros por arquivo:")
display(
    df_ocorrencias_raw
        .select(
            F.col("_metadata.file_path").alias("_source_file")
        )
        .groupBy("_source_file")
        .count()
        .orderBy("_source_file")
)

In [0]:
batch_id_ocorrencias = "ocorrencias_2026_01_05_batch_001"

df_ocorrencias_bronze = (
    df_ocorrencias_raw
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_ingestion_date",
            F.current_date()
        )
        .withColumn(
            "_batch_id",
            F.lit(batch_id_ocorrencias)
        )
)

print("Schema após adição dos metadados:")
df_ocorrencias_bronze.printSchema()

display(
    df_ocorrencias_bronze.select(
        "id_ocorrencia",
        "_source_file",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_batch_id"
    ).limit(10)
)

In [0]:
tabela_ocorrencias = "coretec_dev.bronze.ocorrencias"

(
    df_ocorrencias_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_ocorrencias)
)

print(f"Tabela criada: {tabela_ocorrencias}")

In [0]:
tabela_ocorrencias = "coretec_dev.bronze.ocorrencias"

# 1. Existência e quantidade
print(
    "Tabela existe:",
    spark.catalog.tableExists(tabela_ocorrencias)
)

print(
    "Quantidade de registros:",
    spark.table(tabela_ocorrencias).count()
)

# 2. Schema persistido
print("\nSchema da tabela persistida:")
spark.table(tabela_ocorrencias).printSchema()

# 3. Registros por arquivo de origem
print("\nQuantidade de registros por arquivo:")
display(
    spark.table(tabela_ocorrencias)
        .groupBy("_source_file")
        .count()
        .orderBy("_source_file")
)

# 4. Detalhes da tabela
print("\nDetalhes da tabela:")
display(
    spark.sql(
        f"DESCRIBE DETAIL {tabela_ocorrencias}"
    ).select(
        "format",
        "name",
        "location",
        "numFiles",
        "sizeInBytes"
    )
)

# 5. Histórico Delta
print("\nHistórico Delta:")
display(
    spark.sql(
        f"DESCRIBE HISTORY {tabela_ocorrencias}"
    ).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)

In [0]:
%pip install openpyxl

In [0]:
import pandas as pd

condominios_path = (
    "/Volumes/coretec_dev/landing/coretec_files/"
    "condominios/original/cadastros_smartgate.xlsx"
)

arquivo_excel = pd.ExcelFile(
    condominios_path,
    engine="openpyxl"
)

print("Abas encontradas:", arquivo_excel.sheet_names)

In [0]:
df_condominios_pandas = pd.read_excel(
    condominios_path,
    sheet_name="condominios",
    engine="openpyxl"
)

print("Aba condominios lida com sucesso.")

In [0]:
print("Quantidade de registros:", len(df_condominios_pandas))
print("Quantidade de colunas:", len(df_condominios_pandas.columns))

print("\nNomes das colunas:")
print(df_condominios_pandas.columns.tolist())

print("\nTipos identificados pelo Pandas:")
print(df_condominios_pandas.dtypes)

display(df_condominios_pandas.head(10))

In [0]:
df_condominios_raw = spark.createDataFrame(
    df_condominios_pandas
)

print("DataFrame Spark de condomínios criado com sucesso.")

In [0]:
from pyspark.sql import functions as F

batch_id_condominios = "condominios_batch_001"

df_condominios_bronze = (
    df_condominios_raw
        .withColumn(
            "_source_file",
            F.lit(condominios_path)
        )
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_ingestion_date",
            F.current_date()
        )
        .withColumn(
            "_batch_id",
            F.lit(batch_id_condominios)
        )
)

print("Metadados adicionados ao DataFrame de condomínios.")

In [0]:
print(
    "Quantidade de registros:",
    df_condominios_bronze.count()
)

print("\nSchema após adição dos metadados:")
df_condominios_bronze.printSchema()

display(
    df_condominios_bronze.select(
        "id_condominio",
        "_source_file",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_batch_id"
    ).limit(10)
)

In [0]:
tabela_condominios = "coretec_dev.bronze.condominios"

(
    df_condominios_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_condominios)
)

print(f"Tabela criada: {tabela_condominios}")

In [0]:
tabela_condominios = "coretec_dev.bronze.condominios"

# Existência e quantidade
print(
    "Tabela existe:",
    spark.catalog.tableExists(tabela_condominios)
)

print(
    "Quantidade de registros:",
    spark.table(tabela_condominios).count()
)

# Schema persistido
print("\nSchema da tabela:")
spark.table(tabela_condominios).printSchema()

# Detalhes da tabela Delta
print("\nDetalhes da tabela:")
display(
    spark.sql(
        f"DESCRIBE DETAIL {tabela_condominios}"
    ).select(
        "format",
        "name",
        "location",
        "numFiles",
        "sizeInBytes"
    )
)

# Histórico Delta
print("\nHistórico Delta:")
display(
    spark.sql(
        f"DESCRIBE HISTORY {tabela_condominios}"
    ).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)

In [0]:
tabelas_bronze = [
    "coretec_dev.bronze.acessos_portaria",
    "coretec_dev.bronze.ocorrencias",
    "coretec_dev.bronze.condominios"
]

for tabela in tabelas_bronze:
    detalhes = spark.sql(f"DESCRIBE DETAIL {tabela}").first()
    historico = spark.sql(f"DESCRIBE HISTORY {tabela}").first()

    print(f"Tabela: {tabela}")
    print(f"Registros: {spark.table(tabela).count()}")
    print(f"Formato: {detalhes['format']}")
    print(f"Versão atual: {historico['version']}")
    print("-" * 60)